### Desempenho Operacional por Armazém
**Pergunta de Negócio:** Quais armazéns apresentam maiores percentuais de pedidos atrasados?

**Métricas Calculadas:** Total de pedidos (`total_pedidos`), volume de atrasos (`pedidos_atrasados`) e taxa percentual de atraso (`percentual_atraso`) por armazém.

In [0]:
%sql
SELECT
    warehouse_id AS armazem,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY warehouse_id
ORDER BY percentual_atraso DESC;

**Insight / Resultado:**
Os percentuais de atraso variaram de **54,32% (WH-010)** a **57,59% (WH-008)**, representando uma variação pequena de apenas **3,27 pontos percentuais**. Como todos os armazéns apresentam taxas de atraso altas e muito próximas entre si, o problema não está concentrado em um único centro de distribuição, indicando falhas nos processos operacionais gerais.

### Comparativo de Desempenho por Transportadora
**Pergunta de Negócio:** Quais transportadoras apresentam maior frequência e gravidade de atrasos?

**Métricas Calculadas:** Total de pedidos (`total_pedidos`), volume de atrasos (`pedidos_atrasados`), taxa de atraso (`percentual_atraso`) e média de dias de atraso (`media_dias_atraso`).

In [0]:
%sql
SELECT
    carrier AS transportadora,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso,
    ROUND(
        AVG(CASE
            WHEN late_delivery = 'Yes'
            THEN delivery_delay_days
        END),
        2
    ) AS media_dias_atraso
FROM workspace.default.entregas_tratadas
GROUP BY carrier
ORDER BY percentual_atraso DESC;


**Insight / Resultado:**

A **GlobalExpress** registrou a maior taxa de atraso (**65,75%**) e a maior média de dias de atraso (**2,46 dias**). Em contrapartida, a **SwiftShip** apresentou o melhor desempenho, com **47,34%** de taxa de atraso e média de **2,08 dias**. A diferença entre a pior e a melhor transportadora é de **18,41 pontos percentuais**, sinalizando que a escolha do parceiro logístico impacta diretamente a pontualidade.


###  Isolamento de Variáveis de Confusão (Transportadora vs. Distância e Prazo)
**Pergunta de Negócio:** As diferenças entre as transportadoras permanecem quando comparamos pedidos com faixas de distância e prazos prometidos semelhantes?

**Métricas Calculadas:** Total de pedidos, volume de atrasos e taxa de atraso por combinação de transportadora, faixa de distância e faixa de prazo.

In [0]:
%sql
-- distancia X atrasos
SELECT
    carrier AS transportadora,
    CASE
        WHEN distance_km < 500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END AS faixa_distancia,
    COUNT(*) AS total_pedidos,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    CASE
        WHEN distance_km < 500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END
ORDER BY faixa_distancia, percentual_atraso DESC;

In [0]:
%sql
-- prazo prometido X atrasos

SELECT
    carrier AS transportadora,
    promised_delivery_days AS prazo_prometido_dias,
    COUNT(*) AS total_pedidos,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    promised_delivery_days
ORDER BY
    promised_delivery_days,
    percentual_atraso DESC;

In [0]:
%sql
-- prazo prometido X atrasos

SELECT
    carrier AS transportadora,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END AS faixa_prazo,
    COUNT(*) AS total_pedidos,
    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,
    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END
ORDER BY
    faixa_prazo,
    percentual_atraso DESC;


In [0]:
%sql
--cruzar transportadora, faixa de distância e faixa de prazo prometido.
SELECT
    carrier AS transportadora,

    CASE
        WHEN distance_km < 500 THEN '1. Até 499 km'
        WHEN distance_km < 1000 THEN '2. 500 a 999 km'
        WHEN distance_km < 1500 THEN '3. 1000 a 1499 km'
        ELSE '4. 1500 km ou mais'
    END AS faixa_distancia,

    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END AS faixa_prazo,

    COUNT(*) AS total_pedidos,

    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,

    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso

FROM workspace.default.entregas_tratadas

GROUP BY
    carrier,
    CASE
        WHEN distance_km < 500 THEN '1. Até 499 km'
        WHEN distance_km < 1000 THEN '2. 500 a 999 km'
        WHEN distance_km < 1500 THEN '3. 1000 a 1499 km'
        ELSE '4. 1500 km ou mais'
    END,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END

ORDER BY
    faixa_distancia,
    faixa_prazo,
    percentual_atraso DESC;


**Insight / Resultado:**

Mesmo isolando a distância e o prazo prometido, as variações de desempenho entre os parceiros permanecem. A **GlobalExpress** manteve os maiores índices de atraso na maioria dos cenários (com **21,30%** na faixa mais curta de até 499 km e até 3 dias), enquanto a **SwiftShip** apresentou os menores (**9,87%** no mesmo cenário). Isso confirma que a diferença de desempenho das transportadoras é de origem operacional e não se deve apenas a rotas mais longas.

### Cruzamento Armazém vs. Transportadora
**Pergunta de Negócio:** Existem combinações específicas entre um armazém e uma transportadora que concentrem os piores resultados de atraso?

**Métricas Calculadas:** Total de pedidos (`total_pedidos`), total de atrasos (`pedidos_atrasados`) e taxa de atraso (`percentual_atraso`) por par de armazém e transportadora.

In [0]:
%sql
-- cruzar armazen e transportadora


SELECT
    warehouse_id AS armazem,
    carrier AS transportadora,
    COUNT(*) AS total_pedidos,
    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,
    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    warehouse_id,
    carrier
ORDER BY
    pedidos_atrasados DESC;


**Insight / Resultado:**
A combinação do **WH-001** com a **GlobalExpress** gerou o maior volume de falhas (672 pedidos atrasados e taxa de **69,71%**). A GlobalExpress também lidera o percentual de atrasos no WH-002 (**67,12%**) e no WH-003 (**66,45%**). Os dados indicam que os gargalos se intensificam dependendo do parceiro logístico associado a cada centro de distribuição.